In [1]:
%load_ext autoreload
%autoreload 2

from kontext.visualization import get_figs,plot_boxplots,space_plot,harmonize_target_colors
from kontext.utils import format_key,format_value


from utils.embedding import benchmark_path,run_benchmark_wrapper,run_benchmark
from utils.load_data import get_dataset_info,load_adata
from utils.visualization import plot_sample_heatmap,plot_dataset_rank_dotplot,plot_metric_vs_nobs,readable_sample_name,readable_dataset_name


import pandas as pd 
import submitit
import json
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#fcf8f500'
plt.rcParams['axes.facecolor'] = '#fcf8f500'

from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)

In [2]:
ZESTA = [['ZESTA', 'zf3_1'], ['ZESTA', 'zf5_10'], ['ZESTA', 'zf10_14'], ['ZESTA', 'zf12_8'], ['ZESTA', 'zf12_9'], ['ZESTA', 'zf18_7'], 
         ['ZESTA', 'zf18_8'], ['ZESTA', 'zf18_9'], ['ZESTA', 'zf24_4'], ['ZESTA', 'zf24_7'], ['ZESTA', 'zf24_9'], ['ZESTA', 'zf24_10']]
ARTISTA = [['ARTISTA', '2DPI_1'], ['ARTISTA', '5DPI_2'], ['ARTISTA', '10DPI_3'], ['ARTISTA', '15DPI_1'], 
           ['ARTISTA', '20DPI_2'], ['ARTISTA', '30DPI'], ['ARTISTA', '60DPI']]
MOSTA = [['MOSTA', 'E9.5_E1S1'], ['MOSTA', 'E10.5_E1S1'], ['MOSTA', 'E11.5_E1S1'], 
         ['MOSTA', 'E12.5_E1S1'], ['MOSTA', 'E14.5_E1S1'], ['MOSTA', 'E16.5_E1S1']]
MBA = [['MBA', 'C57BL6J_1_053'], ['MBA', 'C57BL6J_1_082'], ['MBA', 'C57BL6J_1_117'], 
       ['MBA', 'C57BL6J_1_111'], ['MBA', 'C57BL6J_1_113']]

STEREO = ZESTA + ARTISTA + MOSTA 
MERFISH = MBA 
BENCHMARK = STEREO + MERFISH


# 8 - Supp table 
table_1 = []
for dataset,sample in STEREO :
    info = get_dataset_info()[dataset][sample]
#     print(dataset,sample,info['n_obs'],info['n_domains'])
    table_1 += [{'dataset':dataset,'section':sample,'n_cells':info['n_obs'],'total genes':info['n_var'],'n_domains':info['n_domains']}]
df1 = pd.DataFrame(table_1)
print(df1)
# df1.to_csv('./kontext_reproducibility/figures/benchmark/table_stereo_info.csv',index=True)



table_2 = []
for dataset,sample in MERFISH :
    info = get_dataset_info()[dataset][sample]
#     print(dataset,sample,info['n_obs'],info['n_domains'])
    ccf = info['n_domains']['ccf_region_name']
    tissue = info['n_domains']['tissue']
    cell_type = info['n_domains']['cell_type']
    table_2 += [{'dataset':dataset,'section':sample,'n_cells':info['n_obs'],'total genes':info['n_var'],'ccf':ccf,'tissue':tissue,'cell type':cell_type}]
df2 = pd.DataFrame(table_2)
print(df2)
# df2.to_csv('./kontext_reproducibility/figures/benchmark/table_merfish_info.csv',index=True)


## Benchmark stereo-seq datasets 

In [3]:


min_seed,max_seed = 0,100
seeds = range(min_seed,max_seed)
run_all = False # if True, run the benchmark for all the parameters of the loop, otherwise, runs only the parameters for which the result file is missing.  


methods = {
# 'Kontext':{'additional_info':{'name':'Kontext'},'matrix':'Kontext','embedding':{"sigma_e":1, "snn":20, 'alpha':1/2},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'sparse_hvg'}},
'Kontext':{'additional_info':{'name':'Kontext'},'matrix':'Kontext','embedding':{"sigma_e":1, "snn":20, 'alpha':1/2},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'large2svg'}},
'BANKSY':{'matrix':'BANKSY','additional_info':{'name':'BANKSY'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'SEDR':{'matrix':'SEDR','additional_info':{'name':'SEDR gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'SEDR'}},
'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':None}},
}


total_torun = 0
total_done = 0
total_missing = 0
total = 0
results = {}


for dataset,sample in STEREO[:]:
    print(dataset,sample)
    print(dataset,sample,get_dataset_info()[dataset][sample]['n_obs'])
    data = {'dataset':dataset,'sample':sample}
    results[f'{dataset}+{sample}'] = []
    for name,param0 in methods.items():    
            matrix = param0['matrix']
            executor_params = {
                        'slurm_partition':'common' if matrix in ['Kontext','BANKSY'] else 'ml4ig_gpu',
                        'slurm_cpus_per_task':1,
                        'slurm_array_parallelism':100,
                        'slurm_mem':('50G' if dataset=='MOSTA' else '5G') if matrix in ['Kontext','BANKSY'] else ('200G' if dataset=='MOSTA' else '15G'),#get_needed_mem(dataset,sample,step='all'),
                        'timeout_min':600,
                        'slurm_job_name':f'{name}_{sample}',
                        'slurm_srun_args':["--cpu-bind=none",'--nice=19'],
                        'slurm_gres': None if matrix in ['Kontext','BANKSY'] else 'gpu:1',
                        }

            params_all = []
            params_done = []
            params_missing = []

            for seed in seeds:


                param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
                param['data'] = data
                if 'seed' in param['embedding']:
                    param['embedding']['seed'] = seed
                if 'seed' in param['spatial_domains']:
                    param['spatial_domains']['seed'] = seed
                signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],**param['preprocessing'],**param['embedding'],**param['spatial_domains']}.items() if v is not None and k!='additional_info')
                path = benchmark_path(data)
                p = path / f'{signature}.json'
                params_all += [param]
                
                if not p.exists():
                    params_missing += [param]
                else:
                    params_done += [param]
                    with open(p, "r") as f:
                        out = json.load(f)
                    results[f'{dataset}+{sample}'] += [out]


#### Uncomment to run benchmark locally in the loop

                # run_benchmark_wrapper(param)
                # run_example(**param)

####

            executor = submitit.AutoExecutor(folder=f"logs3/benchmark_{dataset}_{sample}/{matrix}/")
            executor.update_parameters(**executor_params ) 

            params_torun = params_all if run_all else params_missing
            n_torun = len(params_all) if run_all else len(params_missing)
            n_done = len(params_done)
            n_missing = len(params_missing)
            total_torun += n_torun
            total_done += n_done
            total_missing += n_missing

            print(f'{dataset} {sample} {matrix} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')

#### Uncomment to submit jobs with submitit

            # print(executor_params)
            # if len(params_torun)>0:
            #     jobs = executor.map_array(run_benchmark_wrapper,params_torun)

#####
print(results)


# Benchmark Mouse Brain Atlas sections

In [4]:

min_seed,max_seed = 0,100
seeds = range(min_seed,max_seed)
run_all = False # if True, run the benchmark for all the parameters of the loop, otherwise, runs only the parameters for which the result file is missing.  
annotations = ['ccf_region_name','tissue','cell_type']

methods = {
'Kontext a=0.0':{'additional_info':{'name':'a=0.0'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.0},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'Kontext a=0.1':{'additional_info':{'name':'a=0.1'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.1},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'Kontext a=0.2':{'additional_info':{'name':'a=0.2'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.2},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'Kontext a=0.3':{'additional_info':{'name':'a=0.3'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.3},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'Kontext a=0.4':{'additional_info':{'name':'a=0.4'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.4},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'Kontext a=0.5':{'additional_info':{'name':'a=0.5'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'Kontext a=0.6':{'additional_info':{'name':'a=0.6'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.6},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'Kontext a=0.7':{'additional_info':{'name':'a=0.7'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.7},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'Kontext a=0.8':{'additional_info':{'name':'a=0.8'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.8},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'Kontext a=0.9':{'additional_info':{'name':'a=0.9'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.9},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'Kontext a=1.0':{'additional_info':{'name':'a=1.0'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':1.0},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},#image'}},
'BANKSY':{'matrix':'BANKSY','additional_info':{'name':'BANKSY'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'SEDR':{'additional_info':{'name':'SEDR gmm'},'matrix':'SEDR',              'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0} ,'preprocessing':{'preprocessing':'SEDR'}},
'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':None}},
}

total_torun = 0
total_done = 0
total_missing = 0
total = 0
# results = {}
nerr = 0
for annotation in  annotations:
    for dataset,sample in MERFISH:
        print(dataset,sample,annotation)
        data = {'dataset':dataset,'sample':sample,'annotation':annotation}
        results[f'{dataset}_{annotation}+{sample}'] = []
        for name,param0 in methods.items():    
            matrix = param0['matrix'] 
            executor_params = {
                        'slurm_partition':'common' if matrix in ['Kontext','BANKSY'] else 'ml4ig_gpu',
                        'slurm_cpus_per_task':1,
                        'slurm_array_parallelism':100,
                        'slurm_mem':'5G' if matrix in ['Kontext','BANKSY'] else '80G',#get_needed_mem(dataset,sample,step='all'),
                        'timeout_min':180,
                        'slurm_job_name':f'{name}_{sample}_{annotation}',
                        'slurm_srun_args':["--cpu-bind=none"],
                        'slurm_gres': None if matrix in ['Kontext','BANKSY'] else 'gpu:1',
                        }

            params_all = []
            params_done = []
            params_missing = []
            for seed in seeds:
                param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
                param['data'] = data
                if matrix == 'Kontext':
                    param['preprocessing']['preprocessing'] = 'xenium'
                if 'seed' in param['embedding']:
                    param['embedding']['seed'] = seed
                if 'seed' in param['spatial_domains']:
                    param['spatial_domains']['seed'] = seed

                signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],**param['preprocessing'],**param['embedding'],**param['spatial_domains']}.items() if v is not None and k!='additional_info')
                path = benchmark_path(data)
                p = path / f'{signature}.json'
                params_all += [param]
                
                if not p.exists():
                    params_missing += [param]
                else:
                    params_done += [param]
                    try:
                        with open(p, "r") as f:
                            out = json.load(f)
                    except:
                        print(p)
                        nerr+=1
                    results[f'{dataset}_{annotation}+{sample}'] += [out]

#### Uncomment to run benchmark locally in the loop
    
                # run_benchmark_wrapper(param)

####



            executor = submitit.AutoExecutor(folder=f"logs3/benchmark_{dataset}_{sample}_{annotation}/{name}/")
            executor.update_parameters(**executor_params ) 

            params_torun = params_all if run_all else params_missing
            n_torun = len(params_all) if run_all else len(params_missing)
            n_done = len(params_done)
            n_missing = len(params_missing)
            total_torun += n_torun
            total_done += n_done
            total_missing += n_missing
            print(f'err{nerr} {dataset} {sample} {annotation} {matrix} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')



#### Uncomment to submit jobs with submitit

            # if len(params_torun)>0:
            #     jobs = executor.map_array(run_benchmark_wrapper,params_torun)
                
#####

print(results)



## Filter and rename the results 

In [5]:
results_heatmap = {}
results_heatmap_stereo = {}
results_heatmap_merfish = {}
results_fixed_alpha = {}
for dataset,sample in BENCHMARK:
    for annotation in annotations:
        # print(sample,annotation)
        if dataset == 'MBA':
            ds = f'{dataset}_{annotation}+{sample}'
        else:
            ds = f'{dataset}+{sample}'
        if ds not in results:
            continue

        res = results[ds]
        new_res = []
        new_res_stereo = []
        new_res_merfish = []
        new_res_fixed_alpha = []

        for run in res:
            i = run.copy()
            # i['name'] = str(i['name']).replace(' gmm','')#.replace('a=1','Kontext')
            for method in ['BANKSY','SEDR','GraphST']:
                if method in i['name']:
                    i['name'] = method
            # i['name'] = str(i['name']).replace('sparse_hvg pp','Kontext').replace(' gmm','')#.replace('a=1','Kontext')
            # i['name'] = str(i['name']).replace(' LR','').replace(' gmm','')#.replace('a=1','Kontext')
            i['dataset'] = f'{dataset} {annotation}'.replace('MBA_','MBA ')
            if 'alpha' in i and dataset == 'MBA' and i['alpha'] == 0.5:
                i_ = i.copy()
                i_['name'] = 'Kontext'
                new_res_fixed_alpha += [i_]

            if 'alpha' in i and dataset=='MBA':
                i['name'] = 'Kontext' if (annotation == 'ccf_region_name' and i['alpha']==0) or (annotation == 'tissue' and i['alpha']==0.1) or (annotation == 'cell_type' and i['alpha']==1) else 'ignore'
                # i['name'] = 'Kontext' if i['alpha'] == 1 else 'other' #(annotation == 'ccf_region_name' and i['alpha']==0) or (annotation == 'tissue' and i['alpha']==0.2) or (annotation == 'cell_type' and i['alpha']==10) else 'other'
            if 'alpha' in i and dataset!='MBA':
                i['name'] = 'Kontext' if i['alpha']==1/2 else 'ignore'
            if i['name'] != 'ignore':
                new_res += [i]
                if 'MBA' in i['dataset']: 
                    new_res_merfish += [i]
                else: 
                    new_res_stereo += [i]
                if i['name'] != 'Kontext':
                    new_res_fixed_alpha += [i]
        results_heatmap[ds] = new_res
        results_heatmap_stereo[ds] = new_res_stereo
        results_heatmap_merfish[ds] = new_res_merfish
        results_fixed_alpha[ds] = new_res_fixed_alpha

# print(results[f'{dataset}+{sample}_{annotation}'])

## Figure  2A 

In [6]:
fig,axes,df1 = plot_dataset_rank_dotplot(results_heatmap,metrics=('ARI','NMI','HOM','COM'),fontsize=12)

def replace(tick):
    replacement = {'MBA_':'','cell_type':'Cell type','tissue':'Tissue','ccf_region_name':'CCF regions'}
    for old,new in replacement.items():
        tick._text = tick._text.replace(old,new)

    return(tick)


yticks = []
axes[0].set_yticklabels([readable_dataset_name(replace(tick).get_text()) for tick in axes[0].get_yticklabels()], rotation=0, fontsize=12) 
for tick in axes[0].get_yticklabels():
    if ('embryo' in tick._text) or ('Axolotl' in tick._text):# ['MOSTA','ARTISTA','ZESTA']:
        # tick.set_text(readable_dataset_name(tick._text)) 
        tick.set_fontweight('bold')
        tick.set_fontsize(15)


for ax in axes:
    title = ax.get_title()
    if title == 'HOM':
        ax.title.set_text('Homogeneity')
    if title == 'COM':
        ax.title.set_text('Completeness')

for ax in axes:
    print(ax.get_title())
plt.tight_layout(rect=[0, 0.08, 1, 1])

gap = 0.05  
for ax in axes[:2]:  
    p = ax.get_position()
    ax.set_position([p.x0 - gap/2, p.y0, p.width, p.height])
for ax in axes[2:]:   
    p = ax.get_position()
    ax.set_position([p.x0 + gap/2, p.y0, p.width, p.height])

# fig.savefig('kontext_project/figures/benchmark/benchmark_1a_ranks.pdf')

## Figure 2B

In [7]:
seed = 0
run_all = False # if True, run the benchmark for all the parameters of the loop, otherwise, runs only the parameters for which the result file is missing.  


methods = {
# 'SEDR':{'matrix':'SEDR','additional_info':{'name':'SEDR'},'embedding':{'seed':0},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':'SEDR'}},
# 'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST'},'embedding':{'seed':0},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':None}},
# 'Kontext mclust':{'matrix':'Kontext','additional_info':{'name':'Kontext mclust'},'embedding':{"sigma_e":0.3, "snn":20, 'alpha':1},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':'sparse_hvg'}},
# 'BANKSY mclust':{'matrix':'BANKSY','additional_info':{'name':'BANKSY mclust'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'Kontext':{'matrix':'Kontext','additional_info':{'name':'sparse_hvg pp'},'embedding':{"sigma_e":1, "snn":20, 'alpha':1/2},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'large2svg'}},
'BANKSY':{'matrix':'BANKSY','additional_info':{'name':'BANKSY'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'SEDR':{'matrix':'SEDR','additional_info':{'name':'SEDR gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'SEDR'}},
'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':None}},
}




TO_SHOW = (
           ['MBA','C57BL6J_1_053'],
     ['ZESTA','zf24_4'],['MOSTA','E11.5_E1S1'],['ARTISTA','5DPI_2'],
           )
annotation = 'ccf_region_name'




ncols = len(methods)+1
nrows = len(TO_SHOW)
fig,axes = plt.subplots(nrows=nrows,ncols=ncols,figsize=(12,12))
i=-1
for dataset,sample in TO_SHOW:
    data = {'dataset':dataset,'sample':sample,'annotation':annotation} if dataset == 'MBA' else {'dataset':dataset,'sample':sample} 
    # dataset = dataset.replace('ZESTA','Zebrafish').replace('MOSTA','Mouse').replace('ARTISTA','Axolotl')
    dataset_name = readable_dataset_name(dataset)
    sample_name = readable_sample_name(sample)
    ylabel = f'{dataset_name}\n{sample_name}'
    # print('ylabel',ylabel)
    # ylabel =f'{dataset} {sample.split('_')[-1]}' if dataset == 'MBA' else f'{dataset} {sample.split('_')[0]}'
    # ylabel = ylabel.replace(' zf24','\nday 24, section 4')
    # ylabel = ylabel.replace(' E11.5','\nembryonic day 11.5')
    # ylabel = ylabel.replace(' 5DPI','\nday 5 after injury')
    # ylabel = ylabel.replace('MBA','Mouse brain atlas \nCommon coordinate framework')
    # ylabel = ylabel.replace(' 053','\nsample 1, section 53')
    
    adata = load_adata(**data)


    i+=1
    j=0
    ax = axes[i,j]
    space_plot(adata,'annotation',s=50,ax=ax)
    ax.get_legend().remove()
    # ax.set_axis_off()

    if i!=0:
        ax.set_title('')
    else:
        ax.set_title(f'Ground Truth',fontsize=20)
    ax.set_xlabel('')
    ax.set_ylabel(f'{ylabel}',fontsize=15)# if dataset == 'MBA' else 15)

    for name,param0 in methods.items():    
            matrix = param0['matrix']
            param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
            if dataset == 'MBA' and name=='Kontext':
                param['preprocessing']={"preprocessing":"xenium"}
                param['embedding']={"sigma_e":1, "snn":20, 'alpha':0}

            param['data'] = data
            signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],
                                                                                  **param['preprocessing'],
                                                                                  **param['embedding'],
                                                                                  **param['spatial_domains']}.items() if v is not None and k!='additional_info')
            path = benchmark_path(data)
            p = path / f'{signature}.json'
            pclust = path / f'{signature}_clustering.csv'

            j+=1
            ax = axes[i,j]
            if pclust.exists():
                df = pd.read_csv(pclust,index_col=0)
                if 'spatial_domains' in df:                 
                    adata.obs[name] = df['spatial_domains']
                else: 
                    adata.obs[name] = df['clustering']

                harmonize_target_colors(adata,col_target=name)
                space_plot(adata,name,s=50,ax=ax)
                ax.get_legend().remove()
                ax.set_xlabel('')
                ax.set_ylabel('')
                if i != 0:
                    ax.set_title('')
                if i == 0:
                    ax.set_title(f'{name}',fontsize=20)
                # ax.set_axis_off()
fig.tight_layout()

plt.show()
# fig.savefig('kontext_project/figures/benchmark/benchmark_1c_space_plots.pdf')
print(results)


## Figure 2C

In [8]:
from kontext.utils import format_size,format_time

metrics = ['embedding_time', 'embedding_memory_b']
fig, axes = get_figs(len(metrics),6,5,vert=True)
for ax, met in zip(axes, metrics):
    print(met)
    plot_metric_vs_nobs(results_heatmap, STEREO, met, ax=ax,logy=True,logx=False)
axes[0].set_title('Computational costs',fontsize=20)
axes[1].set_title('',fontsize=20)
axes[0].set_ylabel('log( embedding time )',fontsize=20)
axes[1].set_ylabel('log( embedding memory )',fontsize=20)
axes[0].set_xlabel('',fontsize=15)
axes[1].set_xlabel('number of cells',fontsize=20)
axes[0].set_xticklabels([])

axes[1].set_xticklabels(['0','0','20k','40k','60k','80k','100k','120k'], rotation='horizontal', fontsize=15) 
axes[1].set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=15) 

times = [1,60,60*5,20*60]
memory_sizes = [10*1024*1024,1024*1024*1024,1024**3*100,1024**3*100*2]

axes[0].set_yticks(times)
axes[0].set_yticklabels([format_time(time).split('-')[0] for time in times],fontsize=15)
axes[1].set_yticks(memory_sizes)
axes[1].set_yticklabels([format_size(memory).replace('.00','') for memory in memory_sizes],fontsize=15)
# axes[0].set_yticklabels([]ax.get_yticklabels(), rotation='horizontal', fontsize=15) 

axes[1].get_legend().remove()

line = axes[1].axhline(1024**3*100*2, linestyle=':', color='red')
axes[1].legend([line], ['Max allocated memory'])
axes[1].set_ylim(memory_sizes[0],1024**3*100*6)
fig.tight_layout()
# fig.savefig('kontext_project/figures/benchmark/benchmark_1b_time_memory.pdf')


## Supplementary figure S1 

In [9]:
nrows = 5
ncols = 5

fig,axes = plt.subplots(nrows=nrows,ncols=ncols,figsize=(4*ncols,4*nrows))
i=-1
for dataset,sample in STEREO:
    data = {'dataset':dataset,'sample':sample,'annotation':annotation} if dataset == 'MBA' else {'dataset':dataset,'sample':sample} 
    dataset_name = readable_dataset_name(dataset)
    sample_name = readable_sample_name(sample)
    title =f'{dataset_name}\n{sample_name}'# {sample.split('_')[-1]}' if dataset == 'MBA' else f'{dataset} {sample.split('_')[0]}'
    # if dataset == 'ZESTA':
    #     title = 'Zebrafish' + sample.replace('zf','\nday ').replace('_',', section ')
    # if dataset == 'MOSTA':
    #     title = 'Mouse' + sample.replace('E1S1','').replace('E','\nembryonic day ')
    # if dataset == 'ARTISTA':
    #     title = 'Axolotl\n'+ sample.replace('DPI','days post injury')


    # ylabel = ylabel.replace('MBA','Mouse brain atlas \nCommon coordinate framework')
    # ylabel = ylabel.replace(' 053','\nsample 1, section 53')
    
    adata = load_adata(**data)


    i+=1
    ax = axes[i//5,i%5]
    space_plot(adata,'annotation',s=20,ax=ax)
    ax.get_legend().remove()
    # title = 'Common coordinate framework' if 'ccf' in annotation else annotation 
    ax.set_title(f'{title}',fontsize=20)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_axis_off()
fig.tight_layout()

## Supplementary figure S2

In [10]:
nrows = 3
ncols = 5

fig,axes = plt.subplots(nrows=nrows,ncols=ncols,figsize=(4*ncols,4*nrows))
i=-1
for annotation in annotations:
    for dataset,sample in MERFISH:
        
        data = {'dataset':dataset,'sample':sample,'annotation':annotation} if dataset == 'MBA' else {'dataset':dataset,'sample':sample} 
        ylabel =f'{dataset} {sample.split('_')[-1]}' if dataset == 'MBA' else f'{dataset} {sample.split('_')[0]}'
        

        # ylabel = ylabel.replace('MBA','Mouse brain atlas \nCommon coordinate framework')
        # ylabel = ylabel.replace(' 053','\nsample 1, section 53')
        
        adata = load_adata(**data)


        i+=1
        ax = axes[i//5,i%5]
        space_plot(adata,'annotation',s=20,ax=ax)
        ax.get_legend().remove()
        if i%5 == 2: 
            title = 'Common coordinate framework' if 'ccf' in annotation else \
                    'Tissue' if annotation == 'tissue' else 'Cell type'
        else : 
            title = ''
        ax.set_title(f'{title}',fontsize=20)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_axis_off()
        

## Supplementary figure S3

In [11]:
fig,axes,df2 = plot_sample_heatmap(results_heatmap, metrics = ('ARI','NMI','HOM','COM'),
                                   agg='std', rect_width=0.85, rect_height=0.20,fontsize_ticks=10,fontsize_scores=9,
                                   lower_better=('ARI','NMI','HOM','COM'),)
fig.tight_layout()

## Supplementary figure S4

In [12]:
fig,axes,df2 = plot_sample_heatmap(results_heatmap, metrics = ('ARI','NMI' ,'HOM','COM'
                                                                          ), agg='mean', rect_width=0.85, rect_height=0.20,fontsize_ticks=10,fontsize_scores=9)


## Supplementary figure S5

In [13]:
supp_results_alpha = {}
min_seed,max_seed = 0,100
seeds = range(min_seed,max_seed)
run_all = False # if True, run the benchmark for all the parameters of the loop, otherwise, runs only the parameters for which the result file is missing.  
annotations = ['ccf_region_name','tissue','cell_type']

total_torun = 0
total_done = 0
total_missing = 0
total = 0
nerr = 0

methods = {
'Kontext alpha=0.0':    {'additional_info':{'name':'alpha=0.0'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.0},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext alpha=0.1':    {'additional_info':{'name':'alpha=0.1'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.1},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext alpha=0.2':    {'additional_info':{'name':'alpha=0.2'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.2},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext alpha=0.3':    {'additional_info':{'name':'alpha=0.3'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.3},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext alpha=0.4':    {'additional_info':{'name':'alpha=0.4'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.4},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext alpha=0.5':    {'additional_info':{'name':'alpha=0.5'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext alpha=0.6':    {'additional_info':{'name':'alpha=0.6'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.6},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext alpha=0.7':    {'additional_info':{'name':'alpha=0.7'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.7},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext alpha=0.8':    {'additional_info':{'name':'alpha=0.8'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.8},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext alpha=0.9':    {'additional_info':{'name':'alpha=0.9'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.9},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext alpha=1.0':    {'additional_info':{'name':'alpha=1.0'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':1.0},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},

# 'BANKSY':{'matrix':'BANKSY','additional_info':{'name':'BANKSY'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'SEDR':{'matrix':'SEDR','additional_info':{'name':'SEDR gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'SEDR'}},
'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':None}},
}





for dataset,sample in BENCHMARK[:]:
    # print(dataset,sample)
    i = -1
    for annotation in annotations: 
        i+=1
        if dataset == 'MBA':
            data = {'dataset':dataset,'sample':sample,'annotation':annotation}
            results_str = f'{dataset}_{annotation}+{sample}'
            supp_results_alpha[results_str] = []
        else:
            if i == 0:
                data = {'dataset':dataset,'sample':sample}
                results_str = f'{dataset}+{sample}'
                supp_results_alpha[results_str] = []

        if dataset == 'MBA' or i == 0:
            for name,param0 in methods.items():  
                if dataset =='MBA':
                    print(results_str,dataset,sample,annotation,name,get_dataset_info()[dataset][sample]['n_obs'])
                else:
                    print(results_str,dataset,sample,name,get_dataset_info()[dataset][sample]['n_obs'])
                matrix = param0['matrix']

                if matrix == 'Kontext':
                    snn1000 =  param0['embedding']['snn']==1000

                matrix = param0['matrix'] 
                executor_params = {
                            'slurm_partition':'common' if matrix in ['Kontext','BANKSY'] else 'ml4ig_gpu',
                            'slurm_cpus_per_task':1,
                            'slurm_array_parallelism':100,
                            'slurm_mem':'100G' if snn1000 else ('100G' if dataset=='MOSTA' else '5G') if matrix in ['Kontext','BANKSY'] else ('200G' if dataset=='MOSTA' else '15G'),#get_needed_mem(dataset,sample,step='all'),
                            'timeout_min':600,
                            'slurm_job_name':f'{name}_{sample}_{annotation}' if dataset == 'MBA' else f'{name}_{sample}',
                            'slurm_srun_args':["--cpu-bind=none"],
                            'slurm_gres': None if matrix in ['Kontext','BANKSY'] else 'gpu:1',
                            }


                params_all = []
                params_done = []
                params_missing = []

                for seed in seeds:


                    param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
                    if matrix == 'Kontext':
                        if dataset == 'MBA':
                            param['preprocessing']['preprocessing'] = 'xenium'
                        else:
                            param['preprocessing']['preprocessing'] = 'large2svg'  

                    param['data'] = data
                    if 'seed' in param['embedding']:
                        param['embedding']['seed'] = seed
                    if 'seed' in param['spatial_domains']:
                        param['spatial_domains']['seed'] = seed
                    if dataset == 'MBA' and matrix == 'Kontext' and 'alpha' not in name:
                        param['embedding']['alpha'] = {'ccf_region_name':0,'tissue':0.1,'cell_type':1}[annotation]

                    signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],**param['preprocessing'],**param['embedding'],**param['spatial_domains']}.items() if v is not None and k!='additional_info')
                    path = benchmark_path(data)
                    p = path / f'{signature}.json'
                    params_all += [param]
                    
                    if not p.exists():
                        print(p)
                        params_missing += [param]
                    else:
                        params_done += [param]
                        try:
                            with open(p, "r") as f:
                                out = json.load(f)
                        except:
                            print(p)
                            nerr+=1

                        out['name'] = name
                        supp_results_alpha[results_str] += [out]





    #### Uncomment to run benchmark locally in the loop

                    # run_benchmark_wrapper(param)
                    # run_example(**param)

    ####        

                if dataset == 'MBA':
                    log_name = f"logs4/benchmark_{dataset}_{sample}_{annotation}/{name}/"
                else:
                    log_name = f"logs4/benchmark_{dataset}_{sample}/{matrix}/"

                executor = submitit.AutoExecutor(folder=log_name)
                executor.update_parameters(**executor_params ) 

                params_torun = params_all if run_all else params_missing
                n_torun = len(params_all) if run_all else len(params_missing)
                n_done = len(params_done)
                n_missing = len(params_missing)
                total_torun += n_torun
                total_done += n_done
                total_missing += n_missing
                if n_missing>0:
                    if dataset == 'MBA':
                        print(f'err{nerr} {dataset} {sample} {annotation} {matrix} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')
                    else:
                        print(f'err{nerr} {dataset} {sample} {matrix} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')
                # else:
                    # print(results_str,len(supp_results_alpha[results_str]))
    #### Uncomment to submit jobs with submitit

                # print(executor_params)
                # if len(params_torun)>0:
                #     jobs = executor.map_array(run_benchmark_wrapper,params_torun)

    #####
    

    print('####',dataset,sample,name,len(supp_results_alpha[results_str]))


###
###
###







In [14]:
results_heatmap_supp_alpha = {}

for dataset,sample in BENCHMARK:
    for annotation in annotations:
        if dataset == 'MBA':
            ds = f'{dataset}_{annotation}+{sample}'
        else:
            ds = f'{dataset}+{sample}'
        # print(ds)
        if ds not in supp_results_alpha:
            print(ds, 'not there')
            continue

        res = supp_results_alpha[ds]
        new_res_supp_alpha = []

        for run in res:
            i = run.copy()
            i['dataset'] = f'{dataset} {annotation}'.replace('MBA_','MBA ')
            i['dataset'] = readable_dataset_name(i['dataset'])
            i['sample'] = readable_sample_name(i['sample'])
            i['name'] = i['name'].replace('Kontext ','')


            if 'alpha' in i['name'] or 'a=' in i['name']: 
                if 'alpha' not in i['name']:
                    i['name'] = i['name'].replace('a=','alpha=')
                new_res_supp_alpha += [i]
            if i['matrix'] != 'Kontext':
                new_res_supp_alpha += [i]
            
            

        results_heatmap_supp_alpha[ds] = new_res_supp_alpha

# print(results_heatmap_supp_sigma)
# fig.savefig('kontext_project/figures/benchmark/benchmark_s1_heatmap.pdf')
# print(results[f'{dataset}+{sample}_{annotation}'])

In [15]:
fig,axes,df2 = plot_sample_heatmap(results_heatmap_supp_alpha, metrics = ('ARI','NMI'
                                                                        # ,'HOM','COM'
                                                                        ), agg='mean', rect_width=0.85, rect_height=0.20,fontsize_ticks=10,fontsize_scores=9)

In [16]:
results_heatmap_supp_alpha = {}

for dataset,sample in BENCHMARK:
    for annotation in annotations:
        if dataset == 'MBA':
            ds = f'{dataset}_{annotation}+{sample}'
        else:
            ds = f'{dataset}+{sample}'
        # print(ds)
        if ds not in supp_results_alpha:
            print(ds, 'not there')
            continue

        res = supp_results_alpha[ds]
        new_res_supp_alpha = []

        for run in res:
            i = run.copy()
            if i['matrix'] != 'Kontext':
                i['name'] = i['matrix']
            else:
                i['name'] = f"alpha = {i['alpha']}"
                # print(i['name'],i['dataset'],i['sample'],i['annotation'],i['alpha'],i['signature'])
            new_res_supp_alpha += [i]


            # i['dataset'] = f'{dataset} {annotation}'.replace('MBA_','MBA ')
            # i['dataset'] = readable_dataset_name(i['dataset'])
            # i['sample'] = readable_sample_name(i['sample'])
            # i['name'] = i['name'].replace('Kontext ','')

            
            

        results_heatmap_supp_alpha[ds] = new_res_supp_alpha

# print(results_heatmap_supp_sigma)
# fig.savefig('kontext_project/figures/benchmark/benchmark_s1_heatmap.pdf')
# print(results[f'{dataset}+{sample}_{annotation}'])


In [17]:
fig,axes,df2 = plot_sample_heatmap(results_heatmap_supp_alpha, metrics = ('ARI','NMI'
                                                                        # ,'HOM','COM'
                                                                        ), agg='mean', rect_width=0.85, rect_height=0.20,fontsize_ticks=10,fontsize_scores=9)

## Supplementary figure S18

In [18]:



min_seed,max_seed = 0,100
# min_seed,max_seed = 10,30
seeds = range(min_seed,max_seed)
run_all = False # if True, run the benchmark for all the parameters of the loop, otherwise, runs only the parameters for which the result file is missing.  


methods = {
'BANKSY l=0.0':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=0.0'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.0, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'BANKSY l=0.1':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=0.1'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.1, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'BANKSY l=0.2':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=0.2'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.2, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'BANKSY l=0.3':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=0.3'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.3, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'BANKSY l=0.4':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=0.4'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'BANKSY l=0.5':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=0.5'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.5, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'BANKSY l=0.6':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=0.6'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.6, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'BANKSY l=0.7':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=0.7'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.7, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'BANKSY l=0.8':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=0.8'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.8, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'BANKSY l=0.9':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=0.9'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.9, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'BANKSY l=1':{'matrix':'BANKSY','additional_info':{'name':'BANKSY l=1'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 1, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'Kontext':{'matrix':'Kontext','additional_info':{'name':'Kontext'},'embedding':{"sigma_e":1, "snn":20, 'alpha':1/2},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'large2svg'}},
}


total_torun = 0
total_done = 0
total_missing = 0
total = 0
results_banksy = {}


# gmm gpu : 18

for dataset,sample in BENCHMARK[:]:
    print(dataset,sample,get_dataset_info()[dataset][sample]['n_obs'])
    i=-1
    for annotation in  annotations:
        i+=1
        if dataset == 'MBA':
            print(dataset,sample,annotation)
            data = {'dataset':dataset,'sample':sample,'annotation':annotation}
            results_str = f'{dataset}_{annotation}+{sample}'
            results_banksy[results_str] = []

        elif i == 0:
            data = {'dataset':dataset,'sample':sample}
            results_str = f'{dataset}_+{sample}'
            results_banksy[results_str] = []

        if dataset == 'MBA' or i==0:
            for name,param0 in methods.items():    
            # if get_dataset_info()[dataset][sample]['n_obs'] > 3000 : 
            # if sample in ['E12.5_E1S1','E14.5_E1S1','E16.5_E1S1']:
                matrix = param0['matrix']

                executor_params = {
                            'slurm_partition':'common' if matrix in ['Kontext','BANKSY'] else 'ml4ig_gpu',
                            'slurm_cpus_per_task':1,
                            'slurm_array_parallelism':100,
                            'slurm_mem':('50G' if dataset=='MOSTA' else '5G') if matrix in ['Kontext','BANKSY'] else ('100G' if dataset=='MOSTA' else '5G'),#get_needed_mem(dataset,sample,step='all'),
                            'timeout_min':600,
                            'slurm_job_name':f'{name}_{sample}',
                            'slurm_srun_args':["--cpu-bind=none",'--nice=19'],
                            'slurm_gres': None if matrix in ['Kontext','BANKSY'] else 'gpu:1',
                            }

                params_all = []
                params_done = []
                params_missing = []


                for seed in seeds:


                    param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
                    param['data'] = data

                    if 'seed' in param['embedding']:
                        param['embedding']['seed'] = seed
                    if 'seed' in param['spatial_domains']:
                        param['spatial_domains']['seed'] = seed
                    if matrix == 'Kontext':
                        param['preprocessing']['preprocessing'] = 'xenium' if dataset == 'MBA' else 'large2svg'
                        if dataset == 'MBA':
                            param['embedding']['alpha'] = {'ccf_region_name':0,'tissue':0.1,'cell_type':1.0}[annotation]
                            
                    # print('param',param)
                    # param = {{'data':data,'method':method,'embedding':embedding,'clustering':clustering,'preprocessing':preprocessing}}
                    signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],**param['preprocessing'],**param['embedding'],**param['spatial_domains']}.items() if v is not None and k!='additional_info')
                    # print('signature',signature)
                    path = benchmark_path(data)
                    p = path / f'{signature}.json'

                    params_all += [param]
                    
                    if not p.exists():
                        params_missing += [param]
                        print(p)
                        # run_benchmark_wrapper(param)
                    else:
                        params_done += [param]
                        with open(p, "r") as f:
                            out = json.load(f)
                        results_banksy[results_str] += [out]
                        # print(out)

                    # run_example(**param)

                if dataset == 'MBA':
                    executor = submitit.AutoExecutor(folder=f"logs/benchmark_{dataset}_{sample}_{annotation}/{matrix}/")
                else:
                    executor = submitit.AutoExecutor(folder=f"logs/benchmark_{dataset}_{sample}/{matrix}/")
                executor.update_parameters(**executor_params ) 

                params_torun = params_all if run_all else params_missing
                n_torun = len(params_all) if run_all else len(params_missing)
                n_done = len(params_done)
                n_missing = len(params_missing)
                total_torun += n_torun
                total_done += n_done
                total_missing += n_missing

                print(f'{dataset} {sample} {name} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')
                # # print(executor_params)
                # if len(params_torun)>0:
                #     jobs = executor.map_array(run_benchmark_wrapper,params_torun)


print(results_banksy[results_str])


In [19]:
banksy_with_kontext = {}
banksy_without_kontext = {}

for k,v in results_banksy.items():
    res_w = []
    res_wo = []
    for i in v:
        # print(i['name'],i['dataset'],i['sample'],i['annotation'],i['ARI'])
        if 'BANKSY' in i['name']:
            i['name'] = f"BANKSY lambda={i['param_lambda']}"
            res_w += [i]
            res_wo += [i]
        if i['matrix']=='Kontext':
            i['name'] = 'Kontext'
            res_w += [i]
            print(i['dataset'],i['sample'],i['annotation'],i['alpha'],i['signature'])
    banksy_with_kontext[k] = res_w
    banksy_without_kontext[k] = res_wo


In [20]:
plot_sample_heatmap(banksy_with_kontext, metrics = ('ARI','NMI'), agg='mean', rect_width=0.85, rect_height=0.20,perfect = False)
plot_sample_heatmap(banksy_without_kontext, metrics = ('ARI','NMI'), agg='mean', rect_width=0.85, rect_height=0.20,perfect = False)

## supplementary figure S19

In [21]:

results_fixed_alpha = {}
for dataset,sample in BENCHMARK:
    for annotation in annotations:
        if dataset == 'MBA':
            ds = f'{dataset}_{annotation}+{sample}'
        else:
            ds = f'{dataset}+{sample}'
        if ds not in results:
            continue

        res = results[ds]
        new_res_fixed_alpha = []

        names = {}
        for run in res:
            i = run.copy()
            for method in ['BANKSY','SEDR','GraphST']:
                if method == i['matrix']:
                    i['name'] = method
                    if i['name'] not in names:
                        print(i['name'],end=' ')
                        names[i['name']] = 0
                    else:
                        names[i['name']]+=1            
                    new_res_fixed_alpha += [i]


            if i['matrix'] == 'Kontext':
                i['name'] = 'Kontext'
                if i['alpha'] == 0.5 and i['snn']==20 and i['sigma_e']==1:
                    # print(i)
                    if i['name'] not in names:
                        print(i['name'],end=' ')
                        names[i['name']] = 0
                    else:
                        names[i['name']]+=1            
                    new_res_fixed_alpha += [i]


        print(names)
        print(ds,len(new_res_fixed_alpha))
        names = {}
        for i in new_res_fixed_alpha:
            if i['matrix'] not in names:
                names[i['matrix']] = 0
            else:
                names[i['matrix']] +=1
        print(names)
        # for i in new_res_fixed_alpha:
        #     print(i['alpha'])
        results_fixed_alpha[ds] = new_res_fixed_alpha


In [22]:
print(results_fixed_alpha[ds][:10])

In [23]:

fig,axes,df2 = plot_sample_heatmap(results_fixed_alpha, metrics = ('ARI','NMI','HOM','COM'),
                                   agg='mean', rect_width=0.85, rect_height=0.20,fontsize_ticks=10,fontsize_scores=9,perfect =True)
fig.tight_layout()

## Supplementary figures S20 

In [24]:
supp_results_snn = {}
min_seed,max_seed = 0,100
seeds = range(min_seed,max_seed)
run_all = False # if True, run the benchmark for all the parameters of the loop, otherwise, runs only the parameters for which the result file is missing.  
annotations = ['ccf_region_name','tissue','cell_type']

total_torun = 0
total_done = 0
total_missing = 0
total = 0

methods = {
'Kontext snn=5':    {'additional_info':{'name':'snn=5'},'matrix':'Kontext',        'embedding':{"sigma_e":1, "snn":5, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}      ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext snn=10':   {'additional_info':{'name':'snn=10'},'matrix':'Kontext',       'embedding':{"sigma_e":1, "snn":10, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}     ,'preprocessing':{'preprocessing':'image'}},
'Kontext snn=20':   {'additional_info':{'name':'snn=20'},'matrix':'Kontext',       'embedding':{"sigma_e":1, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}     ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext snn=30':   {'additional_info':{'name':'snn=30'},'matrix':'Kontext',       'embedding':{"sigma_e":1, "snn":30, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}     ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext snn=50':   {'additional_info':{'name':'snn=50'},'matrix':'Kontext',       'embedding':{"sigma_e":1, "snn":50, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}     ,'preprocessing':{'preprocessing':'image'}},
'Kontext snn=100':  {'additional_info':{'name':'snn=100'},'matrix':'Kontext',      'embedding':{"sigma_e":1, "snn":100, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}    ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext snn=200':  {'additional_info':{'name':'snn=200'},'matrix':'Kontext',      'embedding':{"sigma_e":1, "snn":200, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}    ,'preprocessing':{'preprocessing':'image'}},
'Kontext snn=300':  {'additional_info':{'name':'snn=300'},'matrix':'Kontext',      'embedding':{"sigma_e":1, "snn":300, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}    ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext snn=500':  {'additional_info':{'name':'snn=500'},'matrix':'Kontext',      'embedding':{"sigma_e":1, "snn":500, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}    ,'preprocessing':{'preprocessing':'image'}},
'Kontext snn=1000': {'additional_info':{'name':'snn=1000'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":1000, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'BANKSY':{'matrix':'BANKSY','additional_info':{'name':'BANKSY'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'SEDR':{'matrix':'SEDR','additional_info':{'name':'SEDR gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'SEDR'}},
'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':None}},
}



for dataset,sample in BENCHMARK[:]:
    # print(dataset,sample)
    i = -1
    for annotation in annotations: 
        i+=1
        if dataset == 'MBA':
            data = {'dataset':dataset,'sample':sample,'annotation':annotation}
            results_str = f'{dataset}_{annotation}+{sample}'
            supp_results_snn[results_str] = []
        else:
            if i == 0:
                data = {'dataset':dataset,'sample':sample}
                results_str = f'{dataset}+{sample}'
                supp_results_snn[results_str] = []

        if dataset == 'MBA' or i == 0:
            for name,param0 in methods.items():  
                if dataset =='MBA':
                    print(results_str,dataset,sample,annotation,name,get_dataset_info()[dataset][sample]['n_obs'])
                else:
                    print(results_str,dataset,sample,name,get_dataset_info()[dataset][sample]['n_obs'])
                matrix = param0['matrix']

                if matrix == 'Kontext':
                    snn1000 =  param0['embedding']['snn']==1000

                matrix = param0['matrix'] 
                executor_params = {
                            'slurm_partition':'common' if matrix in ['Kontext','BANKSY'] else 'ml4ig_gpu',
                            'slurm_cpus_per_task':1,
                            'slurm_array_parallelism':100,
                            'slurm_mem':'100G' if snn1000 else ('100G' if dataset=='MOSTA' else '5G') if matrix in ['Kontext','BANKSY'] else ('200G' if dataset=='MOSTA' else '15G'),#get_needed_mem(dataset,sample,step='all'),
                            'timeout_min':600,
                            'slurm_job_name':f'{name}_{sample}_{annotation}' if dataset == 'MBA' else f'{name}_{sample}',
                            'slurm_srun_args':["--cpu-bind=none"],
                            'slurm_gres': None if matrix in ['Kontext','BANKSY'] else 'gpu:1',
                            }


                params_all = []
                params_done = []
                params_missing = []

                for seed in seeds:


                    param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
                    if matrix == 'Kontext':
                        if dataset == 'MBA':
                            param['preprocessing']['preprocessing'] = 'xenium'
                        else:
                            param['preprocessing']['preprocessing'] = 'large2svg'  

                    param['data'] = data
                    if 'seed' in param['embedding']:
                        param['embedding']['seed'] = seed
                    if 'seed' in param['spatial_domains']:
                        param['spatial_domains']['seed'] = seed
                    if dataset == 'MBA' and matrix == 'Kontext' and 'alpha' not in name:
                        param['embedding']['alpha'] = {'ccf_region_name':0,'tissue':0.1,'cell_type':1}[annotation]

                    signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],**param['preprocessing'],**param['embedding'],**param['spatial_domains']}.items() if v is not None and k!='additional_info')
                    path = benchmark_path(data)
                    p = path / f'{signature}.json'
                    params_all += [param]
                    
                    if not p.exists():
                        # print(p)
                        params_missing += [param]
                    else:
                        params_done += [param]
                        try:
                            with open(p, "r") as f:
                                out = json.load(f)
                        except:
                            print(p)
                            nerr+=1

                        out['name'] = name
                        supp_results_snn[results_str] += [out]





    #### Uncomment to run benchmark locally in the loop

                    # run_benchmark_wrapper(param)
                    # run_example(**param)

    ####        

                if dataset == 'MBA':
                    log_name = f"logs4/benchmark_{dataset}_{sample}_{annotation}/{name}/"
                else:
                    log_name = f"logs4/benchmark_{dataset}_{sample}/{matrix}/"

                executor = submitit.AutoExecutor(folder=log_name)
                executor.update_parameters(**executor_params ) 

                params_torun = params_all if run_all else params_missing
                n_torun = len(params_all) if run_all else len(params_missing)
                n_done = len(params_done)
                n_missing = len(params_missing)
                total_torun += n_torun
                total_done += n_done
                total_missing += n_missing
                if n_missing>0:
                    if dataset == 'MBA':
                        print(f'err{nerr} {dataset} {sample} {annotation} {matrix} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')
                    else:
                        print(f'err{nerr} {dataset} {sample} {matrix} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')
                # else:
                    # print(results_str,len(supp_results_alpha[results_str]))
    #### Uncomment to submit jobs with submitit

                # print(executor_params)
                # if len(params_torun)>0:
                #     jobs = executor.map_array(run_benchmark_wrapper,params_torun)

    #####
    

    print('####',dataset,sample,name,len(supp_results_snn[results_str]))





In [25]:

results_heatmap_supp_snn = {}
results_heatmap_supp_snn_time = {}

for dataset,sample in BENCHMARK:
    for annotation in annotations:
        if dataset == 'MBA':
            ds = f'{dataset}_{annotation}+{sample}'
        else:
            ds = f'{dataset}+{sample}'
        if ds not in supp_results_snn:
            continue

        res = supp_results_snn[ds]
        new_res_supp_sigma = []
        new_res_supp_snn   = []
        new_res_supp_snn_time   = []
        new_res_supp_alpha = []

        for run in res:
            i = run.copy()
            i['dataset'] = f'{dataset} {annotation}'.replace('MBA_','MBA ')
            i['dataset'] = readable_dataset_name(i['dataset'])
            i['sample'] = readable_sample_name(i['sample'])
            i['name'] = i['name'].replace('Kontext ','')

            if 'snn' in i['name']:
                # i['name'] = i['name'].replace('snn',r'\ell')
                new_res_supp_snn   += [i]
                
                if sample == 'E16.5_E1S1':
                    new_res_supp_snn_time += [i]
            if i['matrix'] != 'Kontext':
                new_res_supp_snn   += [i]


        results_heatmap_supp_snn[ds] = new_res_supp_snn
        results_heatmap_supp_snn_time[ds] = new_res_supp_snn_time


In [26]:
plt.rcParams['text.usetex'] = True
fig,axes,df2 = plot_sample_heatmap(results_heatmap_supp_snn, metrics = ('ARI','NMI'
                                                                        # ,'HOM','COM'
                                                                        ), agg='mean', rect_width=0.85, rect_height=0.20,fontsize_ticks=10,fontsize_scores=9)
# axes.set_yticklabels([readable_sample_name(sample._text) for sample in axes.get_yticklabels()])
# fig.suptitle('Effect of snn')dra

# axes.set_yticklabels([readable_sample_name(sample._text) for sample in axes.get_yticklabels()])
# fig.suptitle('Effect of alpha')
fig.tight_layout()

## Supplementary figures S21

In [ ]:
supp_results_se = {}
min_seed,max_seed = 0,100
seeds = range(min_seed,max_seed)
run_all = False # if True, run the benchmark for all the parameters of the loop, otherwise, runs only the parameters for which the result file is missing.  
annotations = ['ccf_region_name','tissue','cell_type']

total_torun = 0
total_done = 0
total_missing = 0
total = 0

methods = {
# 'Kontext sigma_e=0.0':    {'additional_info':{'name':'sigma_e=0.0'},'matrix':'Kontext',     'embedding':{"sigma_e":0.0, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
'Kontext sigma_e=0.2':    {'additional_info':{'name':'sigma_e=0.2'},'matrix':'Kontext',     'embedding':{"sigma_e":0.2, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext sigma_e=0.4':    {'additional_info':{'name':'sigma_e=0.4'},'matrix':'Kontext',     'embedding':{"sigma_e":0.4, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext sigma_e=0.6':    {'additional_info':{'name':'sigma_e=0.6'},'matrix':'Kontext',     'embedding':{"sigma_e":0.6, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext sigma_e=0.8':    {'additional_info':{'name':'sigma_e=0.8'},'matrix':'Kontext',     'embedding':{"sigma_e":0.8, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext sigma_e=1.0':    {'additional_info':{'name':'sigma_e=1.0'},'matrix':'Kontext',     'embedding':{"sigma_e":1.0, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext sigma_e=1.2':    {'additional_info':{'name':'sigma_e=1.2'},'matrix':'Kontext',     'embedding':{"sigma_e":1.2, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext sigma_e=1.4':    {'additional_info':{'name':'sigma_e=1.4'},'matrix':'Kontext',     'embedding':{"sigma_e":1.4, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext sigma_e=1.6':    {'additional_info':{'name':'sigma_e=1.6'},'matrix':'Kontext',     'embedding':{"sigma_e":1.6, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext sigma_e=1.8':    {'additional_info':{'name':'sigma_e=1.8'},'matrix':'Kontext',     'embedding':{"sigma_e":1.8, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'Kontext sigma_e=2.0':    {'additional_info':{'name':'sigma_e=2.0'},'matrix':'Kontext',     'embedding':{"sigma_e":2.0, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
'BANKSY':{'matrix':'BANKSY','additional_info':{'name':'BANKSY'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'SEDR':{'matrix':'SEDR','additional_info':{'name':'SEDR gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'SEDR'}},
'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':None}},
}



for dataset,sample in BENCHMARK[:]:
    # print(dataset,sample)
    i = -1
    for annotation in annotations: 
        i+=1
        if dataset == 'MBA':
            data = {'dataset':dataset,'sample':sample,'annotation':annotation}
            results_str = f'{dataset}_{annotation}+{sample}'
            supp_results_se[results_str] = []
        else:
            if i == 0:
                data = {'dataset':dataset,'sample':sample}
                results_str = f'{dataset}+{sample}'
                supp_results_se[results_str] = []

        if dataset == 'MBA' or i == 0:
            for name,param0 in methods.items():  
                if dataset =='MBA':
                    print(results_str,dataset,sample,annotation,name,get_dataset_info()[dataset][sample]['n_obs'])
                else:
                    print(results_str,dataset,sample,name,get_dataset_info()[dataset][sample]['n_obs'])
                matrix = param0['matrix']

                if matrix == 'Kontext':
                    snn1000 =  param0['embedding']['snn']==1000

                matrix = param0['matrix'] 
                executor_params = {
                            'slurm_partition':'common' if matrix in ['Kontext','BANKSY'] else 'ml4ig_gpu',
                            'slurm_cpus_per_task':1,
                            'slurm_array_parallelism':100,
                            'slurm_mem':'100G' if snn1000 else ('100G' if dataset=='MOSTA' else '5G') if matrix in ['Kontext','BANKSY'] else ('200G' if dataset=='MOSTA' else '15G'),#get_needed_mem(dataset,sample,step='all'),
                            'timeout_min':600,
                            'slurm_job_name':f'{name}_{sample}_{annotation}' if dataset == 'MBA' else f'{name}_{sample}',
                            'slurm_srun_args':["--cpu-bind=none"],
                            'slurm_gres': None if matrix in ['Kontext','BANKSY'] else 'gpu:1',
                            }


                params_all = []
                params_done = []
                params_missing = []

                for seed in seeds:


                    param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
                    if matrix == 'Kontext':
                        if dataset == 'MBA':
                            param['preprocessing']['preprocessing'] = 'xenium'
                        else:
                            param['preprocessing']['preprocessing'] = 'large2svg'  

                    param['data'] = data
                    if 'seed' in param['embedding']:
                        param['embedding']['seed'] = seed
                    if 'seed' in param['spatial_domains']:
                        param['spatial_domains']['seed'] = seed
                    if dataset == 'MBA' and matrix == 'Kontext' and 'alpha' not in name:
                        param['embedding']['alpha'] = {'ccf_region_name':0,'tissue':0.1,'cell_type':1.0}[annotation]

                    signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],**param['preprocessing'],**param['embedding'],**param['spatial_domains']}.items() if v is not None and k!='additional_info')
                    path = benchmark_path(data)
                    p = path / f'{signature}.json'
                    params_all += [param]
                    
                    if not p.exists():
                        # print(p)
                        params_missing += [param]
                    else:
                        params_done += [param]
                        try:
                            with open(p, "r") as f:
                                out = json.load(f)
                        except:
                            print(p)
                            nerr+=1

                        out['name'] = name
                        supp_results_se[results_str] += [out]





    #### Uncomment to run benchmark locally in the loop

                    # run_benchmark_wrapper(param)
                    # run_example(**param)

    ####        

                if dataset == 'MBA':
                    log_name = f"logs4/benchmark_{dataset}_{sample}_{annotation}/{name}/"
                else:
                    log_name = f"logs4/benchmark_{dataset}_{sample}/{matrix}/"

                executor = submitit.AutoExecutor(folder=log_name)
                executor.update_parameters(**executor_params ) 

                params_torun = params_all if run_all else params_missing
                n_torun = len(params_all) if run_all else len(params_missing)
                n_done = len(params_done)
                n_missing = len(params_missing)
                total_torun += n_torun
                total_done += n_done
                total_missing += n_missing
                if n_missing>0:
                    if dataset == 'MBA':
                        print(f'err{nerr} {dataset} {sample} {annotation} {matrix} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')
                    else:
                        print(f'err{nerr} {dataset} {sample} {matrix} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')
                # else:
                    # print(results_str,len(supp_results_alpha[results_str]))
    #### Uncomment to submit jobs with submitit

                # print(executor_params)
                if len(params_torun)>0:
                    jobs = executor.map_array(run_benchmark_wrapper,params_torun)

    #####
    

    print('####',dataset,sample,name,len(supp_results_se[results_str]))





In [ ]:

results_heatmap_supp_se = {}


for dataset,sample in BENCHMARK:
    for annotation in annotations:
        if dataset == 'MBA':
            ds = f'{dataset}_{annotation}+{sample}'
        else:
            ds = f'{dataset}+{sample}'
        if ds not in supp_results_se:
            continue

        res = supp_results_se[ds]
        new_res_supp_se = []

        for run in res:
            i = run.copy()
            if i['matrix']!= 'Kontext':
                i['name'] = i['matrix']
                new_res_supp_snn += [i]
            else:
                i['name'] = f"sigma_e = {i['sigma_e']}"
            # i['dataset'] = f'{dataset} {annotation}'.replace('MBA_','MBA ')
            # i['dataset'] = readable_dataset_name(i['dataset'])
            # i['sample'] = readable_sample_name(i['sample'])
            # i['name'] = i['name'].replace('Kontext ','')
            new_res_supp_se +=[i]


        results_heatmap_supp_se[ds] = new_res_supp_se



In [ ]:

fig,axes,df2 = plot_sample_heatmap(results_heatmap_supp_se, metrics = ('ARI','NMI'
                                                                        #   ,'HOM','COM'
                                                                          ), agg='mean', rect_width=0.85, rect_height=0.20,fontsize_ticks=10,fontsize_scores=9)
# axes.set_yticklabels([readable_sample_name(sample._text) for sample in axes.get_yticklabels()])

# fig.suptitle('Effect of sigma_e')

In [ ]:
from utils.visualization import FuncFormatter
metrics = ['embedding_time', 'embedding_memory_b']

df = pd.concat([pd.DataFrame(r).assign(key=k) for k, r in results_heatmap_supp_snn_time.items() if r])

fig, axes = get_figs(len(metrics), vert=True)
for ax, met in zip(axes, metrics):
    for key, sub in df.groupby('key'):
        s = sub.groupby('snn')[met].mean().sort_index()
        ax.plot(s.index, s.values, marker='o', label=key)
    # ax.set_xscale('log') 
    ax.set_yscale('log')
    
    snn = [5,50,100,200,500,1000]
    ax.set_xticks(snn)
    ax.set_xticklabels(snn)
    ax.yaxis.set_major_formatter(FuncFormatter(
        lambda v, _: format_time(v) if 'time' in met else format_size(v)))

axes[0].set_title('Computational costs',fontsize=20)
axes[1].set_title('',fontsize=20)
axes[0].set_ylabel('log( embedding time )',fontsize=20)
axes[1].set_ylabel('log( embedding memory )',fontsize=20)


axes[0].set_xlabel('')
axes[1].set_xlabel('number of neighbors (snn)', fontsize=15)
axes[0].legend(fontsize=8, frameon=False)

times = [1,60,60*5,20*60]
memory_sizes = [10*1024*1024,1024*1024*1024,1024**3*10,1024**3*100,1024**3*100*2]
axes[0].set_yticks(times)
axes[0].set_yticklabels([format_time(time).split('-')[0] for time in times],fontsize=15)
axes[1].set_yticks(memory_sizes)
axes[1].set_yticklabels([format_size(memory).replace('.00','') for memory in memory_sizes],fontsize=15)


fig.tight_layout()
# fig.savefig('kontext_project/figures/benchmark/benchmark_s_snn_time_memory.pdf')

In [ ]:
ctcryo ={
        'unknown':'other',
        'Tumor':'Tumor cell',
        'Plasma_IgG':'Plasma IgG',
        'CAF':'CAF',
        'Fibroblasts':'Fibroblast',
        'T_cell':'T cells',
        'CAM':'CAM',
        'Macrophages':'Macrophage',
        'High_Endothelial_Venules':'other',
        'M1_macrophages':'Macrophage',
        'Plasma_IgA':'Plasma IgA',
        'Germinal_Center_Plasma_IgM_B_cell':'Plasma IgM',
        'Tumor_Keratin_Pearl':'Keratin Pearl',
        'Dendritic_cells':'other',
        'Cytotoxic_IFN_signaling':'other',
        'Mast_cells':'other',
        'Cortex_CCL21':'other'}
ct3D = {
    'CM':'Cardiomyocyte',
    'FB':'Fibroblast',
    'FBmyo':'Myofibroblast',
    'Macro':'Macrophage',
    'mo':'Myeloid_other',
    'DC':'Myeloid_other',
    'Mast':'Myeloid_other',
    'Neutrophil':'Myeloid_other',
    'B':'Lymphoid',
    'ILC2':'Lymphoid',
    'MAIT':'Lymphoid',
    'NK':'Lymphoid',
    'T':'Lymphoid',
    'Treg':'Lymphoid',
    'Fibrocytes':'Fibrocyte',
    'Erythrocyte':'Other',
    'NM':'Other',
    'Schwann':'Other',
    'vEC':'vEC',
    'vEpicardial':'vEpicardial',
    'vPericyte':'vPericyte',
    'vSMC':'vSMC',
    'vSMC1':'vSMC',
    'vSMC2':'vSMC'}

pd.DataFrame(ctcryo.items(),columns = ['original','refined']).to_csv('./kontext_reproducibility/figures/cryo/table_cell_types.csv',index=True)
pd.DataFrame(ct3D.items(),columns=['original','refined']).to_csv('./kontext_reproducibility/figures/lymph/table_cell_types.csv',index=True)

In [ ]:
fig,axes,df2 = plot_sample_heatmap(results_heatmap, metrics = ('CHAOS','PAS'),
                                   lower_better=['CHAOS','PAS'],
                                    agg='mean', rect_width=0.85, rect_height=0.20,fontsize_ticks=10,fontsize_scores=9)


In [ ]:
from utils.load_data import load_adata

from kontext.utils import pca_of_mat

from umap.umap_ import UMAP
from kontext.visualization import umap_plot
from kontext.utils import pca_on_adata

TO_SHOW = (
        #    ['MBA','C57BL6J_1_053'],
     ['ZESTA','zf24_4'],['MOSTA','E11.5_E1S1'],['ARTISTA','5DPI_2'],
           )
annotation = 'ccf_region_name'



i=-1
for dataset,sample in TO_SHOW:
    fig,(ax1,ax2) = get_figs(2)
    data = {'dataset':dataset,'sample':sample,'annotation':annotation} if dataset == 'MBA' else {'dataset':dataset,'sample':sample}     
    adata = load_adata(**data)
    pca_of_mat(adata=adata,mat = adata.X)
    adata.obsm[f'X_umap'] = UMAP(n_components=2).fit_transform(adata.obsm['X_pca'])

    
    umap_plot(adata,color='annotation',ax=ax1,s=10)
    space_plot(adata,color='annotation',ax=ax2,s=10)
    for ax in ax1,ax2:
        ax.get_legend().remove()
        ax.set_xlabel("")
        ax.set_ylabel("")

    ax1.set_title("Expression",fontsize=20)
    ax2.set_title("Spatial positions",fontsize=20)
    plt.show()

In [ ]:




ncols = 1
nrows = 3
for dataset,sample in MERFISH:
    fig,axes = plt.subplots(nrows=nrows,ncols=ncols,figsize=(4*ncols,4*nrows))
    i=-1
    for annotation in annotations: 
        data = {'dataset':dataset,'sample':sample,'annotation':annotation} if dataset == 'MBA' else {'dataset':dataset,'sample':sample} 
        ylabel =f'{dataset} {sample.split('_')[-1]}' if dataset == 'MBA' else f'{dataset} {sample.split('_')[0]}'
        ylabel = ylabel.replace(' zf24','\nday 24, section 4')
        ylabel = ylabel.replace(' E11.5','\nembryonic day 11.5')
        ylabel = ylabel.replace(' 5DPI','\nday 5 after injury')
        ylabel = ylabel.replace('MBA','Mouse brain atlas \nCommon coordinate framework')
        ylabel = ylabel.replace(' 053','\nsample 1, section 53')
        
        adata = load_adata(**data)


        i+=1
        j=0
        ax = axes[i]
        space_plot(adata,'annotation',s=20,ax=ax)
        ax.get_legend().remove()
        title = 'Common coordinate framework' if 'ccf' in annotation else annotation 
        ax.set_title(f'{annotation}',fontsize=20)
        ax.set_xlabel('')
        ax.set_ylabel('')
        # ax.set_ylabel(f'{ylabel}',fontsize=12 if dataset == 'MBA' else 15)

    fig.tight_layout()

    plt.show()
print(results)


In [ ]:
methods = {
'Kontext a=0':{'additional_info':{'name':'a=0'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':0},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=0.1':{'additional_info':{'name':'a=0.1'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':0.1},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=0.2':{'additional_info':{'name':'a=0.2'},'matrix':'Kontext', 'embedding':{"sigma_e":1, "snn":500, 'alpha':0.2},'spatial_domains':{'clustering':'gmm','seed':0} ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=0.3':{'additional_info':{'name':'a=0.3'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':0.3},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=0.4':{'additional_info':{'name':'a=0.4'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':0.4},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=0.5':{'additional_info':{'name':'a=0.5'},'matrix':'Kontext', 'embedding':{"sigma_e":1, "snn":500, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0} ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=1':{'additional_info':{'name':'a=1'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':1},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=2':{'additional_info':{'name':'a=2'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':2},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=3':{'additional_info':{'name':'a=3'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':3},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=10':{'additional_info':{'name':'a=10'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':10},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
'BANKSY':{'additional_info':{'name':'BANKSY'},'matrix':'BANKSY',        'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
# 'SEDR':{'additional_info':{'name':'SEDR'},'matrix':'SEDR',              'embedding':{'seed':0},'spatial_domains':{'clustering':'SEDR_mclust','seed':0} ,'preprocessing':{'preprocessing':'SEDR'}},
# 'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST'},'embedding':{'seed':0},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':None}},
'SEDR gmm':{'additional_info':{'name':'SEDR gmm'},'matrix':'SEDR',              'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0} ,'preprocessing':{'preprocessing':'SEDR'}},
'GraphST gmm':{'matrix':'GraphST','additional_info':{'name':'GraphST gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':None}},
# 'STAGATE':{'embedding':[{'seed':0}],'clustering':[{'clustering':'SEDR_mclust','seed':0}]},
}






# gmm gpu : 18

for dataset,sample in benchmark_m1[:]:
    for annotation in annotations:
        print(dataset,sample,annotation)
        print(dataset,sample,get_dataset_info()[dataset][sample]['n_obs'])
        data = {'dataset':dataset,'sample':sample,'annotation':annotation}
        adata = load_dataset(**data)
        fig,axes = get_figs(len(methods)+1)
        ax = axes[0]
        space_plot(adata,'annotation',s=50,ax=ax)
        ax.get_legend().remove()
        i=0
        for name,param0 in methods.items():    
                matrix = param0['matrix']
                param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
                if matrix == 'Kontext':
                     alpha = 0 if annotation == 'ccf_region_name' else 0.2 if annotation == 'tissue' else 10 
                     param['embedding']['alpha'] = alpha
                     name = f'Kontext a={alpha}'
                param['data'] = data
                signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],**param['preprocessing'],**param['embedding'],**param['spatial_domains']}.items() if v is not None and k!='additional_info')
                path = benchmark_path(data)
                p = path / f'{signature}.json'
                pclust = path / f'{signature}_clustering.csv'

                i+=1
                ax = axes[i]
                if pclust.exists():
                    df = pd.read_csv(pclust,index_col=0)
                    adata.obs[name] = df['spatial_domains']
                    harmonize_target_colors(adata,col_target=name)
                    space_plot(adata,name,s=50,ax=ax)
                    ax.get_legend().remove()

        plt.show()

print(results)

In [ ]:

df1 = plot_dataset_rank_dotplot(results,metrics=('ARI','NMI','HOM','COM'))
df2 = plot_sample_heatmap(results, metrics = ('ARI','NMI','HOM','COM'), agg='mean', rect_width=0.85, rect_height=0.20)

In [ ]:
for dataset,sample in MERFISH:
    print(sample)
    for metric in ['ARI','NMI']:#,'embedding_time','clustering_time']:
        fig,axes = get_figs(3,10,6)
        i=-1
        fig.suptitle(f'{sample} {metric}' )
        for annotation in annotations:
            df = pd.DataFrame(results[f'{dataset}_{annotation}+{sample}'])
            if len(df)>0:
                i+=1
                ax = axes[i]
                plot_boxplots(df, 'name',metric,ax)
                ax.set_title(annotation)
        plt.show()



In [ ]:
for dataset,sample in BENCHMARK:
    print(dataset,sample)
    print(dataset,sample,get_dataset_info()[dataset][sample]['n_obs'])
    data = {'dataset':dataset,'sample':sample}


    adata = load_dataset(**data)
    fig,ax = get_figs(1)
    space_plot(adata,'annotation',s=50,ax=ax)
    ax.set_title(f"{dataset} {sample} n={get_dataset_info()[dataset][sample]['n_obs']}",fontsize=20)
    ax.get_legend().remove()
    plt.show()

In [ ]:
for dataset,sample in MERFISH:
    print(sample)
    fig,axes = get_figs(3)
    i=-1
    fig.suptitle(f'{sample}' )
    adata = load_adata(dataset,sample)
    for annotation in annotations:
        i+=1
        ax = axes[i]
        space_plot(adata,annotation,ax=ax)
        ax.set_title(annotation)
        ax.get_legend().remove()
    plt.show()

## Choose best BANKSY 

## Show a clustering result 

In [ ]:

min_seed,max_seed = 0,1
seeds = range(min_seed,max_seed)


methods = {
'Kontext sparse_hvg':{'matrix':'Kontext','additional_info':{'name':'sparse_hvg pp'},'embedding':{"sigma_e":0.3, "snn":20, 'alpha':1},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'sparse_hvg'}},
# 'BANKSY':{'matrix':'BANKSY','additional_info':{'name':'BANKSY'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
# 'SEDR':{'matrix':'SEDR','additional_info':{'name':'SEDR'},'embedding':{'seed':0},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':'SEDR'}},
# 'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST'},'embedding':{'seed':0},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':None}},
}


total_torun = 0
total_done = 0
total_missing = 0
total = 0
results = {}
def get_figs_col(nrows=4,x=6,y=6):
    fig,axes = plt.subplots(ncols=1,nrows=nrows,figsize=(x,y*nrows))
    return(fig,axes)
 

fig,(ax10,ax20) = get_figs(2)

for dataset,sample in benchmark:
    print(dataset,sample)
    print(dataset,sample,get_dataset_info()[dataset][sample]['n_obs'])
    data = {'dataset':dataset,'sample':sample}
    results[f'{dataset}+{sample}'] = []


    adata = load_adata(**data)
    fig,axes = get_figs_col(len(methods)+1)
    space_plot(adata,'annotation',ax=axes[0],s=50)
    axes[0].get_legend().remove()
    axes[0].set_title('Ground Truth')

    i=0
    for name,param0 in methods.items():    
        # if get_dataset_info()[dataset][sample]['n_obs'] > 3000 : 
        # if sample in ['E12.5_E1S1','E14.5_E1S1','E16.5_E1S1']:
            matrix = param0['matrix']
            seed = 0        


            param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
            param['data'] = data

            if 'seed' in param['embedding']:
                param['embedding']['seed'] = seed
            if 'seed' in param['spatial_domains']:
                param['spatial_domains']['seed'] = seed

            # print('param',param)
            # param = {{'data':data,'method':method,'embedding':embedding,'clustering':clustering,'preprocessing':preprocessing}}
            signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],**param['preprocessing'],**param['embedding'],**param['spatial_domains']}.items() if v is not None and k!='additional_info')
            # print('signature',signature)
            path = benchmark_path(data)
            p = path / f'{signature}.json'

            params_all += [param]
            
            if not p.exists():
                params_missing += [param]
            else:
                params_done += [param]
                with open(p, "r") as f:
                    out = json.load(f)
                results[f'{dataset}+{sample}'] += [out]


            adata = run_example(**param)



            space_plot(adata,'annotation',ax=ax10,s=50)
            space_plot(adata,'spatial_domains',ax=ax20,s=50)
            # plt.show()

            i+=1
            ax = axes[i]
            harmonize_target_colors(adata)
            # s = 50 :
            space_plot(adata,'spatial_domains',ax=ax,s=50)
            ax.get_legend().remove()
            ax.set_title(f'{matrix}')
    plt.show()



In [ ]:




seed = 0
run_all = False # if True, run the benchmark for all the parameters of the loop, otherwise, runs only the parameters for which the result file is missing.  


methods = {
# 'SEDR':{'matrix':'SEDR','additional_info':{'name':'SEDR'},'embedding':{'seed':0},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':'SEDR'}},
# 'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST'},'embedding':{'seed':0},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':None}},
# 'Kontext mclust':{'matrix':'Kontext','additional_info':{'name':'Kontext mclust'},'embedding':{"sigma_e":0.3, "snn":20, 'alpha':1},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':'sparse_hvg'}},
# 'BANKSY mclust':{'matrix':'BANKSY','additional_info':{'name':'BANKSY mclust'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'Kontext':{'matrix':'Kontext','additional_info':{'name':'sparse_hvg pp'},'embedding':{"sigma_e":0.3, "snn":20, 'alpha':1},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'sparse_hvg'}},
'BANKSY':{'matrix':'BANKSY','additional_info':{'name':'BANKSY'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
'SEDR gmm':{'matrix':'SEDR','additional_info':{'name':'SEDR gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'SEDR'}},
'GraphST gmm':{'matrix':'GraphST','additional_info':{'name':'GraphST gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':None}},
}




# gmm gpu : 18

for dataset,sample in benchmark[:]:
    print(dataset,sample)
    print(dataset,sample,get_dataset_info()[dataset][sample]['n_obs'])
    data = {'dataset':dataset,'sample':sample}
    adata = load_adata(**data)
    fig,axes = get_figs(len(methods)+1)
    ax = axes[0]
    space_plot(adata,'annotation',s=50,ax=ax)
    ax.get_legend().remove()
    i=0
    for name,param0 in methods.items():    
            matrix = param0['matrix']

            param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
            param['data'] = data
            signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],
                                                                                    **param['preprocessing'],
                                                                                    **param['embedding'],
                                                                                    **param['spatial_domains']}.items() if v is not None and k!='additional_info')
            path = benchmark_path(data)
            p = path / f'{signature}.json'
            pclust = path / f'{signature}_clustering.csv'

            i+=1
            ax = axes[i]
            if pclust.exists():
                df = pd.read_csv(pclust,index_col=0)
                adata.obs[name] = df['spatial_domains']
                harmonize_target_colors(adata,col_target=name)
                space_plot(adata,name,s=50,ax=ax)
                ax.get_legend().remove()
    plt.show()

print(results)


In [ ]:
methods = {
'Kontext a=0':{'additional_info':{'name':'a=0'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':0},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=0.1':{'additional_info':{'name':'a=0.1'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':0.1},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=0.2':{'additional_info':{'name':'a=0.2'},'matrix':'Kontext', 'embedding':{"sigma_e":1, "snn":500, 'alpha':0.2},'spatial_domains':{'clustering':'gmm','seed':0} ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=0.3':{'additional_info':{'name':'a=0.3'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':0.3},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=0.4':{'additional_info':{'name':'a=0.4'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':0.4},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=0.5':{'additional_info':{'name':'a=0.5'},'matrix':'Kontext', 'embedding':{"sigma_e":1, "snn":500, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0} ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=1':{'additional_info':{'name':'a=1'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':1},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=2':{'additional_info':{'name':'a=2'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':2},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=3':{'additional_info':{'name':'a=3'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':3},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext a=10':{'additional_info':{'name':'a=10'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":500, 'alpha':10},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
'BANKSY':{'additional_info':{'name':'BANKSY'},'matrix':'BANKSY',        'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
# 'SEDR':{'additional_info':{'name':'SEDR'},'matrix':'SEDR',              'embedding':{'seed':0},'spatial_domains':{'clustering':'SEDR_mclust','seed':0} ,'preprocessing':{'preprocessing':'SEDR'}},
# 'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST'},'embedding':{'seed':0},'spatial_domains':{'clustering':'SEDR_mclust','seed':0},'preprocessing':{'preprocessing':None}},
'SEDR gmm':{'additional_info':{'name':'SEDR gmm'},'matrix':'SEDR',              'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0} ,'preprocessing':{'preprocessing':'SEDR'}},
'GraphST gmm':{'matrix':'GraphST','additional_info':{'name':'GraphST gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':None}},
# 'STAGATE':{'embedding':[{'seed':0}],'clustering':[{'clustering':'SEDR_mclust','seed':0}]},
}






# gmm gpu : 18

for dataset,sample in benchmark_m1[:]:
    for annotation in annotations:
        print(dataset,sample,annotation)
        print(dataset,sample,get_dataset_info()[dataset][sample]['n_obs'])
        data = {'dataset':dataset,'sample':sample,'annotation':annotation}
        adata = load_adata(**data)
        fig,axes = get_figs(len(methods)+1)
        ax = axes[0]
        space_plot(adata,'annotation',s=50,ax=ax)
        ax.get_legend().remove()
        i=0
        for name,param0 in methods.items():    
                matrix = param0['matrix']
                param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
                if matrix == 'Kontext':
                     alpha = 0 if annotation == 'ccf_region_name' else 0.2 if annotation == 'tissue' else 10 
                     param['embedding']['alpha'] = alpha
                     name = f'Kontext a={alpha}'
                param['data'] = data
                signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],
                                                                                      **param['preprocessing'],
                                                                                      **param['embedding'],
                                                                                      **param['spatial_domains']}.items() if v is not None and k!='additional_info')
                path = benchmark_path(data)
                p = path / f'{signature}.json'
                pclust = path / f'{signature}_clustering.csv'

                i+=1
                ax = axes[i]
                if pclust.exists():
                    df = pd.read_csv(pclust,index_col=0)
                    adata.obs[name] = df['spatial_domains']
                    harmonize_target_colors(adata,col_target=name)
                    space_plot(adata,name,s=50,ax=ax)
                    ax.get_legend().remove()

        plt.show()

print(results)

In [ ]:
# supp_results = {}
# min_seed,max_seed = 0,100
# seeds = range(min_seed,max_seed)
# run_all = False # if True, run the benchmark for all the parameters of the loop, otherwise, runs only the parameters for which the result file is missing.  
# annotations = ['ccf_region_name','tissue','cell_type']

# total_torun = 0
# total_done = 0
# total_missing = 0
# total = 0

# methods = {
# 'Kontext snn=5':    {'additional_info':{'name':'snn=5'},'matrix':'Kontext',        'embedding':{"sigma_e":1, "snn":5, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}      ,'preprocessing':{'preprocessing':'image'}},
# # 'Kontext snn=10':   {'additional_info':{'name':'snn=10'},'matrix':'Kontext',       'embedding':{"sigma_e":1, "snn":10, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}     ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext snn=20':   {'additional_info':{'name':'snn=20'},'matrix':'Kontext',       'embedding':{"sigma_e":1, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}     ,'preprocessing':{'preprocessing':'image'}},
# # 'Kontext snn=30':   {'additional_info':{'name':'snn=30'},'matrix':'Kontext',       'embedding':{"sigma_e":1, "snn":30, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}     ,'preprocessing':{'preprocessing':'image'}},
# # 'Kontext snn=50':   {'additional_info':{'name':'snn=50'},'matrix':'Kontext',       'embedding':{"sigma_e":1, "snn":50, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}     ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext snn=100':  {'additional_info':{'name':'snn=100'},'matrix':'Kontext',      'embedding':{"sigma_e":1, "snn":100, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}    ,'preprocessing':{'preprocessing':'image'}},
# # 'Kontext snn=200':  {'additional_info':{'name':'snn=200'},'matrix':'Kontext',      'embedding':{"sigma_e":1, "snn":200, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}    ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext snn=300':  {'additional_info':{'name':'snn=300'},'matrix':'Kontext',      'embedding':{"sigma_e":1, "snn":300, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}    ,'preprocessing':{'preprocessing':'image'}},
# # 'Kontext snn=500':  {'additional_info':{'name':'snn=500'},'matrix':'Kontext',      'embedding':{"sigma_e":1, "snn":500, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}    ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext snn=1000': {'additional_info':{'name':'snn=1000'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":1000, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},


# # 'Kontext sigma_e=0.0':    {'additional_info':{'name':'sigma_e=0.0'},'matrix':'Kontext',     'embedding':{"sigma_e":0.0, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'xenium'}},
# 'Kontext sigma_e=0.2':    {'additional_info':{'name':'sigma_e=0.2'},'matrix':'Kontext',     'embedding':{"sigma_e":0.2, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext sigma_e=0.4':    {'additional_info':{'name':'sigma_e=0.4'},'matrix':'Kontext',     'embedding':{"sigma_e":0.4, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext sigma_e=0.6':    {'additional_info':{'name':'sigma_e=0.6'},'matrix':'Kontext',     'embedding':{"sigma_e":0.6, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext sigma_e=0.8':    {'additional_info':{'name':'sigma_e=0.8'},'matrix':'Kontext',     'embedding':{"sigma_e":0.8, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext sigma_e=1.0':    {'additional_info':{'name':'sigma_e=1.0'},'matrix':'Kontext',     'embedding':{"sigma_e":1.0, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext sigma_e=1.2':    {'additional_info':{'name':'sigma_e=1.2'},'matrix':'Kontext',     'embedding':{"sigma_e":1.2, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext sigma_e=1.4':    {'additional_info':{'name':'sigma_e=1.4'},'matrix':'Kontext',     'embedding':{"sigma_e":1.4, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext sigma_e=1.6':    {'additional_info':{'name':'sigma_e=1.6'},'matrix':'Kontext',     'embedding':{"sigma_e":1.6, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext sigma_e=1.8':    {'additional_info':{'name':'sigma_e=1.8'},'matrix':'Kontext',     'embedding':{"sigma_e":1.8, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext sigma_e=2.0':    {'additional_info':{'name':'sigma_e=2.0'},'matrix':'Kontext',     'embedding':{"sigma_e":2.0, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},

# 'Kontext alpha=0.0':    {'additional_info':{'name':'alpha=0.0'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.0},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext alpha=0.1':    {'additional_info':{'name':'alpha=0.1'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.1},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext alpha=0.2':    {'additional_info':{'name':'alpha=0.2'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.2},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext alpha=0.3':    {'additional_info':{'name':'alpha=0.3'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.3},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext alpha=0.4':    {'additional_info':{'name':'alpha=0.4'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.4},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext alpha=0.5':    {'additional_info':{'name':'alpha=0.5'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.5},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext alpha=0.6':    {'additional_info':{'name':'alpha=0.6'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.6},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext alpha=0.7':    {'additional_info':{'name':'alpha=0.7'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.7},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext alpha=0.8':    {'additional_info':{'name':'alpha=0.8'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.8},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext alpha=0.9':    {'additional_info':{'name':'alpha=0.9'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':0.9},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},
# 'Kontext alpha=1.0':    {'additional_info':{'name':'alpha=1.0'},'matrix':'Kontext',     'embedding':{"sigma_e":1, "snn":20, 'alpha':1.0},'spatial_domains':{'clustering':'gmm','seed':0}   ,'preprocessing':{'preprocessing':'image'}},

# # 'BANKSY':{'matrix':'BANKSY','additional_info':{'name':'BANKSY'},'embedding':{'k_geom': 18, 'nbr_weight_decay': 'scaled_gaussian', 'm': 1, 'param_lambda': 0.4, 'n_comps': 50},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'BANKSY'}},
# 'SEDR':{'matrix':'SEDR','additional_info':{'name':'SEDR gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':'SEDR'}},
# 'GraphST':{'matrix':'GraphST','additional_info':{'name':'GraphST gmm'},'embedding':{'seed':0},'spatial_domains':{'clustering':'gmm','seed':0},'preprocessing':{'preprocessing':None}},
# }





# for dataset,sample in STEREO[:]:
#     print(dataset,sample)
#     print(dataset,sample,get_dataset_info()[dataset][sample]['n_obs'])
#     data = {'dataset':dataset,'sample':sample}
#     supp_results[f'{dataset}+{sample}'] = []
#     for name,param0 in methods.items():  
#             print(dataset, sample, name)
#             matrix = param0['matrix']

#             if matrix == 'Kontext':
#                 snn1000 =  param0['embedding']['snn']==1000


#             executor_params = {
#                         'slurm_partition':'common' if matrix in ['Kontext','BANKSY'] else 'ml4ig_gpu',
#                         'slurm_cpus_per_task':1,
#                         'slurm_array_parallelism':100,
#                         'slurm_mem':'100G' if snn1000 else ('100G' if dataset=='MOSTA' else '5G') if matrix in ['Kontext','BANKSY'] else ('200G' if dataset=='MOSTA' else '15G'),#get_needed_mem(dataset,sample,step='all'),
#                         'timeout_min':600,
#                         'slurm_job_name':f'{name}_{sample}',
#                         'slurm_srun_args':["--cpu-bind=none",'--nice=19'],
#                         'slurm_gres': None if matrix in ['Kontext','BANKSY'] else 'gpu:1',
#                         }

#             params_all = []
#             params_done = []
#             params_missing = []

#             for seed in seeds:


#                 param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
#                 if matrix == 'Kontext': 
#                 # param['preprocessing']['preprocessing'] = 'sparse_hvg' 
#                     param['preprocessing']['preprocessing'] = 'large2svg'  

#                 param['data'] = data
#                 if 'seed' in param['embedding']:
#                     param['embedding']['seed'] = seed
#                 if 'seed' in param['spatial_domains']:
#                     param['spatial_domains']['seed'] = seed



#                 signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],**param['preprocessing'],**param['embedding'],**param['spatial_domains']}.items() if v is not None and k!='additional_info')
#                 path = benchmark_path(data)
#                 p = path / f'{signature}.json'
#                 params_all += [param]
                
#                 if not p.exists():
#                     print(p)
#                     params_missing += [param]
#                 else:
#                     params_done += [param]
#                     with open(p, "r") as f:
#                         out = json.load(f)
#                     out['name'] = name
#                     supp_results[f'{dataset}+{sample}'] += [out]


# #### Uncomment to run benchmark locally in the loop

#                 # run_benchmark_wrapper(param)
#                 # run_example(**param)

# ####

#             executor = submitit.AutoExecutor(folder=f"logs4/benchmark_{dataset}_{sample}/{matrix}/")
#             executor.update_parameters(**executor_params ) 

#             params_torun = params_all if run_all else params_missing
#             n_torun = len(params_all) if run_all else len(params_missing)
#             n_done = len(params_done)
#             n_missing = len(params_missing)
#             total_torun += n_torun
#             total_done += n_done
#             total_missing += n_missing
#             if n_missing>0:
#                 print(f'{dataset} {sample} {matrix} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')

# #### Uncomment to submit jobs with submitit

#             # print(executor_params)
#             # if len(params_torun)>0:
#             #     jobs = executor.map_array(run_benchmark_wrapper,params_torun)

# #####
# print(results)



# total_torun = 0
# total_done = 0
# total_missing = 0
# total = 0
# # results = {}
# nerr = 0
# for annotation in  annotations:
#     for dataset,sample in MERFISH:
#         print(dataset,sample,annotation)
#         data = {'dataset':dataset,'sample':sample,'annotation':annotation}
#         supp_results[f'{dataset}_{annotation}+{sample}'] = []
#         for name,param0 in methods.items(): 
#             print(annotation, dataset, sample, name)   
#             matrix = param0['matrix'] 
#             executor_params = {
#                         'slurm_partition':'common' if matrix in ['Kontext','BANKSY'] else 'ml4ig_gpu',
#                         'slurm_cpus_per_task':1,
#                         'slurm_array_parallelism':100,
#                         'slurm_mem':'5G' if matrix in ['Kontext','BANKSY'] else '80G',#get_needed_mem(dataset,sample,step='all'),
#                         'timeout_min':180,
#                         'slurm_job_name':f'{name}_{sample}_{annotation}',
#                         'slurm_srun_args':["--cpu-bind=none"],
#                         'slurm_gres': None if matrix in ['Kontext','BANKSY'] else 'gpu:1',
#                         }

#             params_all = []
#             params_done = []
#             params_missing = []
#             for seed in seeds:
#                 param = {k:v.copy() if isinstance(v,dict) else v for k,v in param0.items()}
#                 param['data'] = data
#                 if 'seed' in param['embedding']:
#                     param['embedding']['seed'] = seed
#                 if 'seed' in param['spatial_domains']:
#                     param['spatial_domains']['seed'] = seed
#                 if 'alpha' not in name:
#                     param['embedding']['alpha'] = {'ccf_region_name':0,'tissue':0.1,'cell_type':1}[annotation]


#                 signature = "_".join(f"{format_key(k)}{format_value(v)}" for k, v in {'matrix':param['matrix'],**param['preprocessing'],**param['embedding'],**param['spatial_domains']}.items() if v is not None and k!='additional_info')
#                 path = benchmark_path(data)
#                 p = path / f'{signature}.json'
#                 params_all += [param]
                
#                 if not p.exists():
#                     params_missing += [param]
#                 else:
#                     params_done += [param]
#                     try:
#                         with open(p, "r") as f:
#                             out = json.load(f)
#                     except:
#                         print(p)
#                         nerr+=1
#                     # print(out)
#                     out['name'] = name
#                     supp_results[f'{dataset}_{annotation}+{sample}'] += [out]

# #### Uncomment to run benchmark locally in the loop
    
#                 # run_benchmark_wrapper(param)

# ####



#             executor = submitit.AutoExecutor(folder=f"logs4/benchmark_{dataset}_{sample}_{annotation}/{name}/")
#             executor.update_parameters(**executor_params ) 

#             params_torun = params_all if run_all else params_missing
#             n_torun = len(params_all) if run_all else len(params_missing)
#             n_done = len(params_done)
#             n_missing = len(params_missing)
#             total_torun += n_torun
#             total_done += n_done
#             total_missing += n_missing
#             if n_missing>0:
#                 print(f'err{nerr} {dataset} {sample} {annotation} {matrix} to_run={n_torun} done={n_done} missing={n_missing} total_torun={total_torun} total_done={total_done} total_missing={total_missing}')



# #### Uncomment to submit jobs with submitit

#             # if len(params_torun)>0:
#             #     jobs = executor.map_array(run_benchmark_wrapper,params_torun)
                
# #####

# print(supp_results)



In [ ]:
# results_heatmap_supp_sigma = {}
# results_heatmap_supp_snn = {}
# results_heatmap_supp_alpha = {}
# results_heatmap_supp_snn_time = {}

# for dataset,sample in BENCHMARK:
#     for annotation in annotations:
#         if dataset == 'MBA':
#             ds = f'{dataset}_{annotation}+{sample}'
#         else:
#             ds = f'{dataset}+{sample}'
#         if ds not in supp_results:
#             continue

#         res = supp_results[ds]
#         new_res_supp_sigma = []
#         new_res_supp_snn   = []
#         new_res_supp_snn_time   = []
#         new_res_supp_alpha = []

#         for run in res:
#             i = run.copy()
#             i['dataset'] = f'{dataset} {annotation}'.replace('MBA_','MBA ')
#             i['dataset'] = readable_dataset_name(i['dataset'])
#             i['sample'] = readable_sample_name(i['sample'])
#             i['name'] = i['name'].replace('Kontext ','')
#             if 'alpha' in i['name'] or 'a=' in i['name']: 
#                 if 'alpha' not in i['name']:
#                     i['name'] = i['name'].replace('a=','alpha=')
#                 new_res_supp_alpha += [i]
#             if 'snn' in i['name']:
#                 new_res_supp_snn   += [i]
#                 if sample == 'E16.5_E1S1':
#                     new_res_supp_snn_time += [i]
#             if 'sigma' in i['name']:
#                 new_res_supp_sigma += [i]
#             if i['matrix'] != 'Kontext':
#                 new_res_supp_snn   += [i]
#                 new_res_supp_alpha += [i]
#                 new_res_supp_sigma += [i]

#             # if i['name'] == '<Kontext':
#             #     i_alpha = i.copy()
#             #     i_alpha['name'] = 'alpha=0.5'
#             #     i_snn = i.copy()
#             #     i_snn['name'] = 'snn=20'
#             #     i_sigma = i.copy()
#             #     i_sigma['name'] = 'sigma_e=1.0'

#             #     new_res_supp_alpha += [i_alpha]
#             #     new_res_supp_snn   += [i_snn]
#             #     new_res_supp_sigma += [i_sigma]
#             # if 'snn' not in i['name'] and 'alpha' not in i['name'] and 'sigma' not in i['name'] and 'a=' not in i['name'] and i['name']!='Kontext':
#             #     print(i['dataset'],i['name'],i)
#             # print(dataset,i['snn'],i['name'])
#             # if dataset == 'MBA' and i['snn'] == 'snn=20' and i['name'] == 'Kontext':
#             #     print(i)
            
#         results_heatmap_supp_sigma[ds] = new_res_supp_sigma
#         results_heatmap_supp_snn[ds] = new_res_supp_snn
#         results_heatmap_supp_snn_time[ds] = new_res_supp_snn_time
#         results_heatmap_supp_alpha[ds] = new_res_supp_alpha

# # print(results_heatmap_supp_sigma)
# # fig.savefig('kontext_project/figures/benchmark/benchmark_s1_heatmap.pdf')
# # print(results[f'{dataset}+{sample}_{annotation}'])